In [3]:
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.portfolio import generate_portfolio

from src.mortality import (
    load_ssa_tables,
    prepare_life_table,
    combine_life_tables,
    merge_expected_mortality,
)

from src.experience import (
    assign_mortality_multiplier,
    simulate_deaths,
    summarize_experience,
)

In [5]:
# Generate portfolio
portfolio = generate_portfolio()

# Load mortality tables
female_raw, male_raw = load_ssa_tables()

# Prepare mortality tables
female = prepare_life_table(female_raw)
male = prepare_life_table(male_raw)

# Combine male and female tables
life_table = combine_life_tables(
    male,
    female
)

# Add expected mortality to portfolio
portfolio = merge_expected_mortality(
    portfolio,
    life_table
)

In [7]:
portfolio = assign_mortality_multiplier(portfolio)

portfolio = simulate_deaths(portfolio)

In [8]:
#Summarizing Experience

experience = (
    portfolio
    .groupby("attained_age")
    .agg(
        expected_deaths=("expected_deaths", "sum"),
        actual_deaths=("death", "sum"),
        exposure=("policy_id", "count")
    )
    .reset_index()
)

experience["AE"] = (
    experience["actual_deaths"] /
    experience["expected_deaths"]
)

experience.head()

,attained_age,expected_deaths,actual_deaths,exposure,AE
0,21,0.006704,0,7,0.0
1,22,0.012542,0,11,0.0
2,23,0.015645,0,15,0.0
3,24,0.028824,0,24,0.0
4,25,0.037268,0,31,0.0


In [ ]:
#Group ages into bands to make AE ratio more interpretible
bins = [20, 30, 40, 50, 60, 70, 80, 90, 101]
labels = [
    "20–29",
    "30–39",
    "40–49",
    "50–59",
    "60–69",
    "70–79",
    "80–89",
    "90–100"
]

portfolio["age_band"] = pd.cut(
    portfolio["attained_age"],
    bins=bins,
    labels=labels,
    right=False
)

band_experience = (
    portfolio
    .groupby("age_band")
    .agg(
        exposure=("policy_id", "count"),
        expected=("expected_deaths", "sum"),
        actual=("death", "sum")
    )
)

band_experience["AE"] = (
    band_experience["actual"] /
    band_experience["expected"]
)

band_experience

Visualizations and Conclusions:

In [ ]:
plot_expected_vs_actual(experience)

plot_ae_by_age(experience)

plot_ae_by_age_band(band_experience)

Then Testing

In [ ]:
test_portfolio = pd.DataFrame({
    "policy_id": range(1,1001),
    "qx": 0.0,
    "mortality_multiplier":1.0
})

test_result = simulate_deaths(test_portfolio)

print("Numebr of deaths:", test_result["death"].sum())